# Laya on CUAD: 41 yes/no clause questions

Standalone Colab version of the `jev-bench` Laya run. It needs no repo checkout.

[Laya](https://huggingface.co/convaiinnovations/laya) is a 421M-param non-autoregressive decision model (ModernBERT-large, 512-token window). It is **not** a chat LLM. Each CUAD category is asked as a `noul` (yes/no) question, and the answer is Yes when `P(true) >= THRESHOLD`.

Each contract is split into overlapping chunks, and a question counts as Yes if **any** chunk crosses the threshold (`chunk_any_yes`, same as the repo). The notebook keeps the **max** P(true) per question, so the threshold can be swept afterwards without re-running the model.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
%pip install -q "laya>=0.3.20" huggingface_hub scikit-learn pandas tqdm

In [ ]:
import json, random, time
from pathlib import Path

import torch, laya
print("laya", laya.__version__, "| torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 1. Smoke test

In [ ]:
agent = laya.load("convaiinnovations/laya", device="cuda" if torch.cuda.is_available() else "cpu")

state = ("Either party may terminate this Agreement for convenience upon thirty (30) days' prior written "
         "notice to the other party. This Agreement shall be governed by the laws of the State of Delaware.")
questions = {
    "termination_for_convenience": {"type": "noul", "instructions":
        'Does this contract contain a "Termination For Convenience" clause?'},
    "non_compete": {"type": "noul", "instructions":
        'Does this contract contain a "Non-Compete" clause?'},
}
res = agent.predict(state, questions)
for k, v in res["answers"].items():
    print(f"{k:30s} P(yes)={v['noul']:.3f}")

## 2. Config

In [ ]:
N_CONTRACTS = 20        # out of 510; None = all (a T4 takes hours for the full set)
SEED = 42
THRESHOLD = 0.5         # Yes when max-chunk P(true) >= THRESHOLD
MAX_CHARS = 1200        # ~320 tokens of contract fits beside the question in the 512-token window
CHUNK_OVERLAP = 200
CHUNK_BATCH = 8         # chunks per forward pass; each chunk has 41 question rows, so 8 -> 328 rows
OUT = Path("/content/laya_cuad"); OUT.mkdir(parents=True, exist_ok=True)

## 3. Download and parse CUAD v1

`datasets.load_dataset("theatticusproject/cuad")` doesn't give the SQuAD JSON, so the raw file is fetched directly.

In [ ]:
from huggingface_hub import hf_hub_download

cuad_path = hf_hub_download("theatticusproject/cuad", "CUAD_v1/CUAD_v1.json", repo_type="dataset")
raw = json.load(open(cuad_path, encoding="utf-8"))

def clean_description(q):
    return q.split("Details:", 1)[1].strip() if "Details:" in q else q.strip()

contracts = []  # [{"title", "context", "qs": [{"category", "instructions", "label"}]}]
for doc in raw["data"]:
    para = doc["paragraphs"][0]
    qs = []
    for qa in para["qas"]:
        cat = qa["id"].split("__")[-1]
        qs.append({
            "category": cat,
            # Same text as the repo's YES_NO_TEMPLATE minus the trailing "Answer Yes or No."
            "instructions": f'Does this contract contain a "{cat}" clause? CUAD definition: {clean_description(qa["question"])}',
            "label": not qa.get("is_impossible", len(qa.get("answers", [])) == 0),
        })
    contracts.append({"title": doc["title"], "context": para["context"], "qs": qs})

print(len(contracts), "contracts,", sum(len(c["qs"]) for c in contracts), "examples")

## 4. Run

Resumable: finished contracts are appended to `preds.jsonl` and skipped on a rerun.

In [ ]:
from tqdm.auto import tqdm

def chunk_text(text, max_chars, overlap):
    if len(text) <= max_chars:
        return [text]
    step = max(1, max_chars - overlap)
    return [text[i:i + max_chars] for i in range(0, len(text), step) if text[i:i + max_chars].strip()]

subset = contracts if N_CONTRACTS is None else random.Random(SEED).sample(contracts, N_CONTRACTS)
preds_path = OUT / "preds.jsonl"
done = set()
if preds_path.exists():
    done = {json.loads(l)["contract_id"] for l in open(preds_path, encoding="utf-8")}

with open(preds_path, "a", encoding="utf-8") as f:
    for c in tqdm([c for c in subset if c["title"] not in done], desc="contracts"):
        chunks = chunk_text(c["context"], MAX_CHARS, CHUNK_OVERLAP)
        questions = {q["category"]: {"type": "noul", "instructions": q["instructions"]} for q in c["qs"]}
        t0 = time.perf_counter()
        results = agent.predict_batch(chunks, questions, batch_size=CHUNK_BATCH)
        latency = time.perf_counter() - t0
        for q in c["qs"]:
            probs = [r["answers"][q["category"]]["noul"] for r in results]
            f.write(json.dumps({
                "id": f'{c["title"]}__{q["category"]}', "contract_id": c["title"], "category": q["category"],
                "label": q["label"], "p_max": max(probs), "n_chunks": len(chunks),
                "latency_s": latency / len(c["qs"]),
            }) + "\n")
        f.flush()

## 5. Metrics

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

wanted = {c["title"] for c in subset}
df = pd.read_json(preds_path, lines=True)
df = df[df.contract_id.isin(wanted)].drop_duplicates("id")

def prf(y, p):
    pr, rc, f1, _ = precision_recall_fscore_support(y, p, average="binary", zero_division=0)
    return {"n": len(y), "accuracy": accuracy_score(y, p), "precision": pr, "recall": rc, "f1": f1}

df["pred"] = df.p_max >= THRESHOLD
per_cat = pd.DataFrame({cat: prf(g.label, g.pred) for cat, g in df.groupby("category")}).T
summary = {
    "model": "convaiinnovations/laya", "threshold": THRESHOLD, "contracts": df.contract_id.nunique(),
    "micro": prf(df.label, df.pred),
    "macro": per_cat[["precision", "recall", "f1"]].mean().to_dict(),
    "mean_latency_s": df.latency_s.mean(), "mean_chunks_per_contract": df.groupby("contract_id").n_chunks.first().mean(),
}
(OUT / "metrics.json").write_text(json.dumps({**summary, "per_category": per_cat.to_dict("index")}, indent=2, default=float))
print(json.dumps(summary, indent=2, default=float))
per_cat.sort_values("f1")

## 6. Threshold sweep

Chunk-level any-yes pushes toward false positives, because a long contract gives many chances to cross the threshold. The best threshold is usually well above 0.5.

In [ ]:
sweep = pd.DataFrame([{"threshold": t, **prf(df.label, df.p_max >= t)} for t in [x / 20 for x in range(1, 20)]])
best = sweep.loc[sweep.f1.idxmax()]
print(f"best micro-F1 {best.f1:.3f} at threshold {best.threshold:.2f}")
sweep.style.format(precision=3).highlight_max(subset=["f1"])

## 7. Save results

Download the outputs, or mount Drive first (`from google.colab import drive; drive.mount('/content/drive')`) and point `OUT` there so a full run survives a disconnect.

In [ ]:
import shutil
shutil.make_archive("/content/laya_cuad_results", "zip", OUT)
try:
    from google.colab import files
    files.download("/content/laya_cuad_results.zip")
except ImportError:
    print("Not on Colab; results in", OUT)